In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy import stats
from scipy.stats import rankdata
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set(font_scale=1.05)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# Find project root
def find_project_root(marker='data'):
    cwd = Path.cwd()
    for path in (cwd, *cwd.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Could not find project root containing '{marker}'")

ROOT = find_project_root()
print(f"Project root: {ROOT}")

Project root: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI


In [2]:
# Metric helpers
def compute_metrics(y_true, y_pred):
    """Compute MAE, RMSE, MAPE, R² for a single series."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

def diebold_mariano_test(actual, pred1, pred2, h=1):
    """Diebold-Mariano test for forecast comparison.
    H0: pred1 and pred2 have equal accuracy.
    Returns: DM statistic, p-value.
    """
    e1 = actual - pred1
    e2 = actual - pred2
    d = (e1 ** 2) - (e2 ** 2)
    dm = np.mean(d) / np.sqrt(np.var(d) / len(d))
    p_val = 2 * (1 - stats.norm.cdf(np.abs(dm)))
    return dm, p_val

horizons = ['t1', 't3', 't5']
horizon_labels = {0: 't1', 1: 't3', 2: 't5'}

print("Metric helpers ready.")

Metric helpers ready.


## Load Ground Truth & Test Dates
Load y_test and dates from 05_ensemble_methods (Hybrid B pipeline).

In [3]:
# Load y_test from 05_ensemble_methods (re-extract from data)
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = ROOT / "notebooks" / "model_ready_dataset.csv"
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

train_end = pd.Timestamp('2017-12-31')
val_end = pd.Timestamp('2021-12-31')
test_mask = df['date'] > val_end
train_mask = df['date'] <= train_end
val_mask = (df['date'] > train_end) & (df['date'] <= val_end)

target_cols = ['cpi_t1', 'cpi_t3', 'cpi_t5']
y_all = df[target_cols].values

scaler_y = MinMaxScaler()
scaler_y.fit(y_all[train_mask.values])
y_test = y_all[test_mask.values]
test_dates = df[test_mask]['date'].values
lookback = 18

# Align to lookback offset (same as in ensemble)
test_dates_aligned = test_dates[lookback:]
y_test_aligned = y_test[lookback:]

print(f"Ground truth shape: {y_test_aligned.shape}")
print(f"Test dates: {len(test_dates_aligned)}")
print(f"Date range: {test_dates_aligned[0]} to {test_dates_aligned[-1]}")

Ground truth shape: (20, 3)
Test dates: 20
Date range: 2023-07-01T00:00:00.000000000 to 2025-02-01T00:00:00.000000000


## Load ARIMA, LSTM, RF Predictions
From notebooks 03, 04 (stored in memory/notebook variables).

In [4]:
# Load ARIMA from 03_baseline_models
# ARIMA forecast for test period: (n_test, 3) for horizons t1, t3, t5
# Note: You need to re-run notebook 03 or extract forecasts manually
# Placeholder: assume test_forecast (n_test, 3) from ARIMA in 03

# For now, load from 03's outputs if available, else reconstruct
arima_preds = None  # placeholder

print("ARIMA predictions: placeholder (load from 03 or re-run forecast)")
print("Recommended: Run ARIMA forecast on test set and store here.")

ARIMA predictions: placeholder (load from 03 or re-run forecast)
Recommended: Run ARIMA forecast on test set and store here.


## Load Hybrid A & Hybrid B Predictions
From saved artifacts in results/ directories.

In [5]:
# Load Hybrid A
hyb_a_dir = ROOT / "results" / "hybrid_a"
hybrid_a_test = np.load(hyb_a_dir / "ensembleA_test_pred.npy")  # (n_test, 3)
hybrid_a_test = hybrid_a_test[lookback:]  # align to lookback offset

# Load Hybrid B
hyb_b_dir = ROOT / "results" / "hybrid_b"
hybrid_b_test = np.load(hyb_b_dir / "ensembleB_test_pred.npy")  # (n_test, 3)
hybrid_b_test = hybrid_b_test[lookback:]  # align to lookback offset

print(f"Hybrid A shape: {hybrid_a_test.shape}")
print(f"Hybrid B shape: {hybrid_b_test.shape}")
print(f"Ground truth shape: {y_test_aligned.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\reset\\OneDrive - UWE Bristol\\Master_project\\UK-inflation-forecasting-XAI\\results\\hybrid_b\\ensembleB_test_pred.npy'

In [ ]:
# LSTM and RF predictions from 04_ml_models
# These are stored in notebook 04, so we'll reference them or save them here
# For now, placeholder structure
lstm_test = None  # placeholder: load from 04's y_test_pred
rf_test = None    # placeholder: load from 04's y_hat_test

print("LSTM & RF: placeholders (load from 04 or modify this cell)")
print("Recommended: Extract y_test_pred (LSTM) and y_hat_test (RF) from 04 and store as .npy")

## Table 1: Forecast Performance Comparison
RMSE (per horizon), MAE, MAPE, R² on test set.

In [ ]:
# Build Table 1: Performance metrics
# Models: ARIMA, LSTM, RF, Hybrid A, Hybrid B
# Metrics: RMSE_t1, RMSE_t3, RMSE_t5, MAE, MAPE, R2

table1_rows = []
model_names = ['Hybrid A', 'Hybrid B']
model_preds = [hybrid_a_test, hybrid_b_test]

for name, preds in zip(model_names, model_preds):
    row = {'Model': name}
    for i, h in enumerate(horizons):
        m = compute_metrics(y_test_aligned[:, i], preds[:, i])
        row[f'RMSE_{h}'] = m['RMSE']
    # Overall metrics (average across horizons)
    m_overall = compute_metrics(y_test_aligned.flatten(), preds.flatten())
    row['MAE'] = m_overall['MAE']
    row['MAPE'] = m_overall['MAPE']
    row['R2'] = m_overall['R2']
    table1_rows.append(row)

table1 = pd.DataFrame(table1_rows)
print("\n=== TABLE 1: Forecast Performance ===")
print(table1.to_string(index=False))
print(f"\nNote: ARIMA, LSTM, RF to be added once extracted from notebooks 03 & 04.")

## Table 2: Feature Importance & Stability
XGBoost gain, SHAP values, rank correlation (stability).

In [ ]:
# Load feature names and XGBoost models for Table 2
with open(hyb_a_dir / "feature_names.json") as f:
    feature_names = json.load(f)

import xgboost as xgb

# Load Hybrid A XGBoost models
xgb_models_a = {}
for h in horizons:
    model_path = hyb_a_dir / f"xgb_{h}.json"
    xgb_models_a[h] = xgb.Booster(model_file=str(model_path))

# Extract feature importance (gain) for t1
importance_dict = xgb_models_a['t1'].get_score(importance_type='gain')
importance_df = pd.DataFrame(list(importance_dict.items()), columns=['Feature', 'Importance']).sort_values('Importance', ascending=False)

print("\n=== TABLE 2 (Partial): Feature Importance (Hybrid A, t1) ===")
print(importance_df.head(10).to_string(index=False))
print("\nNote: Full Table 2 includes SHAP values (from notebook 06) and stability across horizons.")

## Table 3: Statistical Tests (Diebold-Mariano)
Pairwise comparisons of model accuracy on test set.

In [ ]:
# Diebold-Mariano test: Hybrid B vs Hybrid A
table3_rows = []

for i, h in enumerate(horizons):
    dm_stat, p_val = diebold_mariano_test(
        y_test_aligned[:, i],
        hybrid_a_test[:, i],
        hybrid_b_test[:, i]
    )
    significant = 'Yes' if p_val < 0.05 else 'No'
    winner = 'Hybrid B' if dm_stat > 0 else 'Hybrid A'
    table3_rows.append({
        'Comparison': f'Hybrid A vs Hybrid B ({h})',
        'DM Statistic': dm_stat,
        'P-Value': p_val,
        'Significant (α=0.05)': significant,
        'Winner': winner
    })

table3 = pd.DataFrame(table3_rows)
print("\n=== TABLE 3: Diebold-Mariano Tests ===")
print(table3.to_string(index=False))

## Table 4: Period-Based Evaluation
Performance by COVID/Brexit/Post-COVID periods.

In [ ]:
# Define periods
covid_start = pd.Timestamp('2020-03-01')
covid_end = pd.Timestamp('2021-12-31')
brexit_date = pd.Timestamp('2020-01-31')

# Masks for test period
pre_covid_mask = test_dates_aligned < covid_start
covid_mask = (test_dates_aligned >= covid_start) & (test_dates_aligned <= covid_end)
post_covid_mask = test_dates_aligned > covid_end
brexit_mask = (test_dates_aligned >= brexit_date) & (test_dates_aligned < covid_start)

period_masks = {
    'Pre-COVID': pre_covid_mask,
    'Brexit': brexit_mask,
    'COVID': covid_mask,
    'Post-COVID': post_covid_mask
}

table4_rows = []
for period_name, mask in period_masks.items():
    if mask.sum() == 0:
        continue
    y_period = y_test_aligned[mask]
    hyb_a_period = hybrid_a_test[mask]
    hyb_b_period = hybrid_b_test[mask]
    
    m_a = compute_metrics(y_period.flatten(), hyb_a_period.flatten())
    m_b = compute_metrics(y_period.flatten(), hyb_b_period.flatten())
    
    table4_rows.append({
        'Period': period_name,
        'Data Points': mask.sum(),
        'Hybrid A RMSE': m_a['RMSE'],
        'Hybrid B RMSE': m_b['RMSE'],
        'Hybrid A MAE': m_a['MAE'],
        'Hybrid B MAE': m_b['MAE']
    })

table4 = pd.DataFrame(table4_rows)
print("\n=== TABLE 4: Period-Based Evaluation ===")
print(table4.to_string(index=False))

# Part II: Visualizations
Six plots for comprehensive model comparison.

## Plot 1: Time-Series Actual vs Predicted
Overlay predictions from Hybrid A and Hybrid B on actual CPI YoY.

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Time-Series: Actual vs Hybrid A vs Hybrid B Forecasts', fontsize=16, fontweight='bold')

for idx, (ax, h) in enumerate(zip(axes, horizons)):
    ax.plot(test_dates_aligned, y_test_aligned[:, idx], 'o-', label='Actual', linewidth=2.5, markersize=4, color='blue')
    ax.plot(test_dates_aligned, hybrid_a_test[:, idx], 's--', label='Hybrid A', linewidth=2, markersize=3, alpha=0.8, color='red')
    ax.plot(test_dates_aligned, hybrid_b_test[:, idx], '^--', label='Hybrid B', linewidth=2, markersize=3, alpha=0.8, color='green')
    
    ax.set_title(f'Forecast Horizon {h}', fontsize=12, fontweight='bold')
    ax.set_ylabel('CPI YoY (%)', fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(loc='best', frameon=True)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

axes[-1].set_xlabel('Date', fontsize=10)
plt.show()

print("Plot 1: Time-Series comparison rendered.")

## Plot 2: Forecast Error Distribution
Histogram + KDE of forecast errors per model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
fig.suptitle('Forecast Error Distribution (All Horizons)', fontsize=14, fontweight='bold')

errors_a = (y_test_aligned - hybrid_a_test).flatten()
errors_b = (y_test_aligned - hybrid_b_test).flatten()

# Histogram
axes[0].hist(errors_a, bins=20, alpha=0.6, label='Hybrid A', color='red', density=True)
axes[0].hist(errors_b, bins=20, alpha=0.6, label='Hybrid B', color='green', density=True)
axes[0].set_xlabel('Forecast Error (Actual - Predicted)', fontsize=10)
axes[0].set_ylabel('Density', fontsize=10)
axes[0].set_title('Histogram of Errors', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# KDE
from scipy.stats import gaussian_kde
kde_a = gaussian_kde(errors_a)
kde_b = gaussian_kde(errors_b)
x_range = np.linspace(errors_a.min(), errors_a.max(), 100)
axes[1].plot(x_range, kde_a(x_range), label='Hybrid A', linewidth=2.5, color='red')
axes[1].plot(x_range, kde_b(x_range), label='Hybrid B', linewidth=2.5, color='green')
axes[1].fill_between(x_range, kde_a(x_range), alpha=0.3, color='red')
axes[1].fill_between(x_range, kde_b(x_range), alpha=0.3, color='green')
axes[1].set_xlabel('Forecast Error', fontsize=10)
axes[1].set_ylabel('Density', fontsize=10)
axes[1].set_title('KDE of Errors', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.show()
print("Plot 2: Error distribution rendered.")

## Plot 3: Rolling Window RMSE
24-month rolling RMSE to show performance stability over time.

In [ ]:
# Compute rolling RMSE
window_size = min(24, len(y_test_aligned) // 3)  # 24-month window or smaller
step = 6  # 6-month step

rolling_rmse_a = []
rolling_rmse_b = []
rolling_dates = []

for i in range(0, len(y_test_aligned) - window_size + 1, step):
    window_end = i + window_size
    y_win = y_test_aligned[i:window_end].flatten()
    pred_a_win = hybrid_a_test[i:window_end].flatten()
    pred_b_win = hybrid_b_test[i:window_end].flatten()
    
    rolling_rmse_a.append(np.sqrt(mean_squared_error(y_win, pred_a_win)))
    rolling_rmse_b.append(np.sqrt(mean_squared_error(y_win, pred_b_win)))
    rolling_dates.append(test_dates_aligned[window_end - 1])

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
ax.plot(rolling_dates, rolling_rmse_a, 'o-', label='Hybrid A', linewidth=2.5, markersize=6, color='red')
ax.plot(rolling_dates, rolling_rmse_b, 's-', label='Hybrid B', linewidth=2.5, markersize=6, color='green')
ax.fill_between(rolling_dates, rolling_rmse_a, alpha=0.2, color='red')
ax.fill_between(rolling_dates, rolling_rmse_b, alpha=0.2, color='green')

ax.set_title(f'Rolling RMSE (Window: {window_size} months, Step: {step} months)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('RMSE', fontsize=11)
ax.legend(loc='best', frameon=True, fontsize=10)
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.show()
print("Plot 3: Rolling RMSE rendered.")

## Plot 4: Predicted vs Actual (Scatter)
Calibration plot with perfect agreement diagonal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
fig.suptitle('Predicted vs Actual (Test Set)', fontsize=14, fontweight='bold')

for idx, (ax, h) in enumerate(zip(axes, horizons)):
    y_h = y_test_aligned[:, idx]
    pred_a_h = hybrid_a_test[:, idx]
    pred_b_h = hybrid_b_test[:, idx]
    
    ax.scatter(y_h, pred_a_h, alpha=0.6, s=30, label='Hybrid A', color='red', edgecolor='darkred', linewidth=0.5)
    ax.scatter(y_h, pred_b_h, alpha=0.6, s=30, label='Hybrid B', color='green', edgecolor='darkgreen', linewidth=0.5)
    
    # Perfect agreement line
    minv = min(y_h.min(), pred_a_h.min(), pred_b_h.min())
    maxv = max(y_h.max(), pred_a_h.max(), pred_b_h.max())
    ax.plot([minv, maxv], [minv, maxv], 'k--', linewidth=1, label='Perfect Agreement')
    
    ax.set_xlim(minv, maxv)
    ax.set_ylim(minv, maxv)
    ax.set_xlabel('Actual CPI YoY (%)', fontsize=10)
    ax.set_ylabel('Predicted CPI YoY (%)', fontsize=10)
    ax.set_title(f'Horizon {h}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.show()
print("Plot 4: Scatter plot rendered.")

## Plot 5: Feature Importance (XGBoost Hybrid A)
Top 10 features by gain across horizons.

In [ ]:
# Aggregate feature importance across horizons
importance_agg = {}
for h in horizons:
    imp_dict = xgb_models_a[h].get_score(importance_type='gain')
    for feat, score in imp_dict.items():
        importance_agg[feat] = importance_agg.get(feat, 0) + score

importance_sorted = sorted(importance_agg.items(), key=lambda x: x[1], reverse=True)[:10]
features, scores = zip(*importance_sorted)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.barh(features, scores, color='steelblue', edgecolor='black', linewidth=1.2)
ax.set_xlabel('Aggregated Gain', fontsize=11, fontweight='bold')
ax.set_title('Top 10 Features (Hybrid A, XGBoost)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.show()
print("Plot 5: Feature importance rendered.")

## Plot 6: Model Comparison (Metrics by Horizon)
Bar chart of MAE and RMSE per model per horizon.

In [ ]:
# Prepare metrics data
metrics_data = []
for name, preds in [('Hybrid A', hybrid_a_test), ('Hybrid B', hybrid_b_test)]:
    for i, h in enumerate(horizons):
        m = compute_metrics(y_test_aligned[:, i], preds[:, i])
        metrics_data.append({'Model': name, 'Horizon': h, 'MAE': m['MAE'], 'RMSE': m['RMSE']})

metrics_df = pd.DataFrame(metrics_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# MAE by horizon
mae_pivot = metrics_df.pivot(index='Horizon', columns='Model', values='MAE')
mae_pivot.plot(kind='bar', ax=axes[0], color=['red', 'green'], edgecolor='black', linewidth=1.2)
axes[0].set_title('MAE by Horizon', fontsize=12, fontweight='bold')
axes[0].set_ylabel('MAE', fontsize=10)
axes[0].set_xlabel('Horizon', fontsize=10)
axes[0].legend(title='Model', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# RMSE by horizon
rmse_pivot = metrics_df.pivot(index='Horizon', columns='Model', values='RMSE')
rmse_pivot.plot(kind='bar', ax=axes[1], color=['red', 'green'], edgecolor='black', linewidth=1.2)
axes[1].set_title('RMSE by Horizon', fontsize=12, fontweight='bold')
axes[1].set_ylabel('RMSE', fontsize=10)
axes[1].set_xlabel('Horizon', fontsize=10)
axes[1].legend(title='Model', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.show()
print("Plot 6: Model comparison rendered.")

## Notes
- **Plot 2 (SHAP) & Plot 3 (Stability)**: Link to notebook 06 for SHAP analysis and lime explanations.
- **ARIMA, LSTM, RF**: Extract predictions from notebooks 03 & 04 and add to Table 1 for full comparison.
- **Overfitting/Underfitting Detection**: Compare train vs test metrics (not yet included; can add if train preds are saved).
- **Recommendation**: Save all model predictions (ARIMA, LSTM, RF, Hybrid A, Hybrid B) as .npy files for reproducibility.